# 02 — Cleaning

**Goal:** turn the 9 raw CSVs into typed, de-duplicated, English-labelled tables with the derived columns every later notebook and the Power BI model need. Output goes to `data/processed/` and is the single source for the SQL star schema (`sql/02_load_data.sql`).

**Principles**
1. *Never drop information silently.* Rows are removed only when they are exact or logical duplicates; anything suspicious gets a **flag column** instead.
2. *One row per key.* After this notebook every table has a clean primary key, so star-schema joins can't fan out.
3. *Every decision is written down* — in a markdown cell here and in `docs/decision_log.md`.

Input: `data/raw/*.csv`. Output: `data/processed/*.csv` + a before/after row-count table.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 160)
RAW, OUT = Path("../data/raw"), Path("../data/processed")
OUT.mkdir(exist_ok=True)

ZIP_COLS = ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]
read = lambda f: pd.read_csv(RAW / f, dtype={c: str for c in ZIP_COLS})

orders      = read("olist_orders_dataset.csv")
items       = read("olist_order_items_dataset.csv")
customers   = read("olist_customers_dataset.csv")
sellers     = read("olist_sellers_dataset.csv")
products    = read("olist_products_dataset.csv")
payments    = read("olist_order_payments_dataset.csv")
reviews     = read("olist_order_reviews_dataset.csv")
geo         = read("olist_geolocation_dataset.csv")
translation = read("product_category_name_translation.csv")

before = {n: len(df) for n, df in dict(orders=orders, order_items=items, customers=customers, sellers=sellers,
          products=products, payments=payments, reviews=reviews, geolocation=geo).items()}
before

{'orders': 99441,
 'order_items': 112650,
 'customers': 99441,
 'sellers': 3095,
 'products': 32951,
 'payments': 103886,
 'reviews': 99224,
 'geolocation': 1000163}

## 1. Timestamps → datetime

All 8 date/time columns arrive as strings. Converting once here means every downstream notebook and the SQL loader get real timestamps.

In [2]:
ts_cols = {
    "orders":  ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items":   ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
for df, cols in [(orders, ts_cols["orders"]), (items, ts_cols["items"]), (reviews, ts_cols["reviews"])]:
    for c in cols:
        df[c] = pd.to_datetime(df[c], errors="coerce")
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 2. Reviews — one review per order

Phase 1 showed 547 orders with more than one review row and 1,603 rows sharing a `review_id` (the same review text attached to several orders — a survey-system artefact).

**Decision:** keep the **latest answered** review per `order_id`. A customer's final word is the best single signal, and one-row-per-order prevents review counts inflating when joined to items.

In [3]:
reviews = (reviews.sort_values("review_answer_timestamp")
                  .drop_duplicates(subset="order_id", keep="last")
                  .reset_index(drop=True))
assert reviews.order_id.is_unique
print(f"reviews: {before['reviews']:,} -> {len(reviews):,}  (removed {before['reviews']-len(reviews):,} duplicate rows)")

reviews: 99,224 -> 98,673  (removed 551 duplicate rows)


## 3. Geolocation — one coordinate per postcode prefix

1,000,163 rows describe only 19,015 prefixes; 261,831 rows are exact duplicates and the rest are many GPS samples per prefix.

**Decision:** drop exact duplicates, then collapse to the **mean lat/lng and most common city/state per prefix**. This becomes `dim_geography`. We also normalise city names (lower-case, strip accents) so they join cleanly to customer/seller cities.

In [4]:
import unicodedata
def normalise_city(s: pd.Series) -> pd.Series:
    return (s.str.lower().str.strip()
             .apply(lambda x: unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode() if isinstance(x, str) else x))

geo = geo.drop_duplicates()
n_dedup = len(geo)
geo["geolocation_city"] = normalise_city(geo["geolocation_city"])

# Brazil bounding box — a handful of coordinates fall outside it (data-entry errors)
in_brazil = geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)
print(f"coordinates outside Brazil removed: {(~in_brazil).sum():,}")
geo = geo[in_brazil]

mode = lambda s: s.mode().iat[0]
geography = (geo.groupby("geolocation_zip_code_prefix")
                .agg(lat=("geolocation_lat", "mean"), lng=("geolocation_lng", "mean"),
                     city=("geolocation_city", mode), state=("geolocation_state", mode),
                     n_samples=("geolocation_lat", "size"))
                .reset_index().rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"}))
print(f"geolocation: {before['geolocation']:,} -> {n_dedup:,} (exact dups) -> {len(geography):,} (one row per prefix)")
geography.head()

coordinates outside Brazil removed: 33


geolocation: 1,000,163 -> 738,332 (exact dups) -> 19,010 (one row per prefix)


,zip_code_prefix,lat,lng,city,state,n_samples
0,01001,-23.550227,-46.634039,sao paulo,SP,11
1,01002,-23.547657,-46.634991,sao paulo,SP,6
2,01003,-23.549000,-46.635582,sao paulo,SP,11
3,01004,-23.549829,-46.634792,sao paulo,SP,14
4,01005,-23.549547,-46.636406,sao paulo,SP,13


## 4. Products — English categories

Two Portuguese categories are missing from the translation table and 610 products have no category at all.

**Decision:** left-join the translation table, hand-translate the two gaps, and label the nulls `unknown` so they stay in the data rather than disappearing from category charts.

In [5]:
extra = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_preparers"],
})
translation = pd.concat([translation, extra], ignore_index=True)

products = products.merge(translation, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")
products = products.rename(columns={"product_name_lenght": "product_name_length",       # fix source typos
                                    "product_description_lenght": "product_description_length"})
# product volume is useful for the freight analysis later
products["product_volume_cm3"] = products.product_length_cm * products.product_height_cm * products.product_width_cm
assert products.product_category_name_english.isna().sum() == 0
products.product_category_name_english.value_counts().head(10)

product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

## 5. Orders — derived delivery columns

| New column | Definition |
|---|---|
| `delivery_days` | actual delivery date − purchase date (days, float) |
| `estimated_delivery_days` | promised date − purchase date |
| `days_late` | actual − promised (positive = late, negative = early) |
| `is_late` | `days_late > 0` (NaN when there is no delivery date) |
| `order_month` | first day of the purchase month (for time series) |
| `approval_hours` | approval − purchase (payment-approval lag) |
| `carrier_days` | handover to carrier − purchase (seller processing time) |

Promised date is a date without time; we compare at day granularity so a parcel delivered at 18:00 on the promised day is **not** late.

In [6]:
d = orders  # alias
d["order_month"]             = d.order_purchase_timestamp.dt.to_period("M").dt.to_timestamp()
d["delivery_days"]           = (d.order_delivered_customer_date - d.order_purchase_timestamp).dt.total_seconds() / 86400
d["estimated_delivery_days"] = (d.order_estimated_delivery_date - d.order_purchase_timestamp).dt.total_seconds() / 86400
d["days_late"]               = (d.order_delivered_customer_date.dt.normalize() - d.order_estimated_delivery_date.dt.normalize()).dt.days
d["is_late"]                 = d.days_late.gt(0).where(d.days_late.notna())
d["approval_hours"]          = (d.order_approved_at - d.order_purchase_timestamp).dt.total_seconds() / 3600
d["carrier_days"]            = (d.order_delivered_carrier_date - d.order_purchase_timestamp).dt.total_seconds() / 86400
d["is_delivered"]            = d.order_status.eq("delivered")

print(f"delivered orders: {d.is_delivered.sum():,}  |  late (of delivered with a date): {d.is_late.sum():,.0f} "
      f"= {d.loc[d.is_delivered, 'is_late'].mean():.1%}")
d[["delivery_days", "estimated_delivery_days", "days_late", "approval_hours", "carrier_days"]].describe().round(2)

delivered orders: 96,478  |  late (of delivered with a date): 6,535 = 6.8%


,delivery_days,estimated_delivery_days,days_late,approval_hours,carrier_days
count,96476.00,99441.00,96476.00,99281.00,97658.00
mean,12.56,23.77,-11.88,10.42,3.23
std,9.55,8.83,10.18,26.04,3.61
min,0.53,1.65,-147.00,0.00,-171.21
25%,6.77,18.33,-17.00,0.22,1.13
50%,10.22,23.24,-12.00,0.34,2.20
75%,15.72,28.42,-7.00,14.58,4.07
max,209.63,155.14,188.00,4509.18,125.78


## 6. Order items — outlier flags, never drops

7.5% of prices and 10.8% of freight values sit above the 1.5×IQR fence. These are real transactions (a R$6,735 item is a legitimate sale), so dropping them would understate revenue.

**Decision:** add `is_price_outlier` / `is_freight_outlier` booleans so analysts and Power BI can exclude them deliberately when looking at *typical* behaviour.

In [7]:
def iqr_flag(s: pd.Series) -> pd.Series:
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    return (s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)

items["is_price_outlier"]   = iqr_flag(items.price)
items["is_freight_outlier"] = iqr_flag(items.freight_value)
items["item_total"]         = items.price + items.freight_value
items["freight_ratio"]      = items.freight_value / items.price
items[["is_price_outlier", "is_freight_outlier"]].mean().mul(100).round(1).rename("pct_flagged")

is_price_outlier       7.5
is_freight_outlier    10.8
Name: pct_flagged, dtype: float64

## 7. Payments — keep rows, add an order-level summary

Payments are 1-to-many with orders (vouchers + card). We keep the row-level table **and** build `order_payments_summary` (one row per order) so the fact table can carry payment type and instalments without fan-out.

In [8]:
payments["payment_type"] = payments.payment_type.replace("not_defined", "unknown")

main_type = (payments.sort_values("payment_value", ascending=False)
                     .drop_duplicates("order_id")[["order_id", "payment_type", "payment_installments"]]
                     .rename(columns={"payment_type": "main_payment_type", "payment_installments": "max_installments"}))
order_payments_summary = (payments.groupby("order_id")
                                  .agg(payment_total=("payment_value", "sum"), n_payments=("payment_sequential", "size"))
                                  .reset_index().merge(main_type, on="order_id", how="left"))
assert order_payments_summary.order_id.is_unique
order_payments_summary.main_payment_type.value_counts(normalize=True).mul(100).round(1)

main_payment_type
credit_card    75.4
boleto         19.9
voucher         3.2
debit_card      1.5
unknown         0.0
Name: proportion, dtype: float64

## 8. Customers and sellers — light touch

Only normalisation of city strings (lower-case, no accents) to match `geography`. `customer_id` stays as the order-level key; `customer_unique_id` is carried for person-level analysis.

In [9]:
customers["customer_city"] = normalise_city(customers.customer_city)
sellers["seller_city"]     = normalise_city(sellers.seller_city)
customers["is_repeat_customer"] = customers.customer_unique_id.duplicated(keep=False)
print(f"unique people: {customers.customer_unique_id.nunique():,} across {len(customers):,} orders; "
      f"{customers.drop_duplicates('customer_unique_id').is_repeat_customer.mean():.2%} of people ordered more than once")

unique people: 96,096 across 99,441 orders; 3.12% of people ordered more than once


## 9. Export and before/after audit

In [10]:
exports = {
    "orders":                 orders,
    "order_items":            items,
    "customers":              customers,
    "sellers":                sellers,
    "products":               products,
    "payments":               payments,
    "order_payments_summary": order_payments_summary,
    "reviews":                reviews,
    "geography":              geography,
    "category_translation":   translation,
}
for name, df in exports.items():
    df.to_csv(OUT / f"{name}.csv", index=False)

audit = pd.DataFrame({"rows_before": pd.Series(before), "rows_after": {k: len(v) for k, v in exports.items()}})
audit["rows_removed"] = audit.rows_before - audit.rows_after
audit["note"] = ""
audit.loc["reviews", "note"]     = "kept latest review per order"
audit.loc["geolocation", "note"] = "see 'geography' — one row per zip prefix"
audit.loc["geography", "note"]   = "derived from geolocation"
audit.loc["order_payments_summary", "note"] = "derived from payments"
audit.loc["category_translation", "note"]   = "+2 hand-translated categories"
audit

,rows_before,rows_after,rows_removed,note
category_translation,NaN,73.0,NaN,+2 hand-translated categories
customers,99441.0,99441.0,0.0,
geography,NaN,19010.0,NaN,derived from geolocation
geolocation,1000163.0,NaN,NaN,see 'geography' — one row per zip prefix
order_items,112650.0,112650.0,0.0,
order_payments_summary,NaN,99440.0,NaN,derived from payments
orders,99441.0,99441.0,0.0,
payments,103886.0,103886.0,0.0,
products,32951.0,32951.0,0.0,
reviews,99224.0,98673.0,551.0,kept latest review per order


## So what (3 bullets)

- **Nothing of analytical value was discarded.** The only rows removed are duplicate reviews and duplicate/redundant GPS samples; outliers and odd orders are flagged, not deleted, so revenue totals stay honest.
- **Every table now has a clean key** (order_id, (order_id, order_item_id), customer_id, seller_id, product_id, zip prefix), which is what makes the star schema in Phase 2b safe to build with plain joins.
- **The derived delivery columns already hint at the headline story:** a measurable share of delivered orders arrive after the promised date — Phase 3 will test whether that is what drags review scores down.